In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv("exchange_bias_enhanced.csv")

In [8]:
eps = 1e-8

# -------------------------------
# Magnetic interaction descriptors
# -------------------------------

df["magmom_mean"] = (
    df["core_magmom"] +
    df["shell_magmom"]
) / 2

df["magmom_norm_diff"] = (
    df["magmom_diff"] /
    (df["magmom_mean"] + eps)
)

df["magmom_ofm"] = (
    df["magmom_product"] *
    df["ofm_cosine"]
)

df["magmom_distance"] = (
    df["magmom_product"] *
    (1 - df["ofm_cosine"])
)

# -------------------------------
# Orbital interaction descriptors
# -------------------------------

df["d_occ_product"] = (
    df["core_d_occ"] *
    df["shell_d_occ"]
)

df["f_occ_product"] = (
    df["core_f_occ"] *
    df["shell_f_occ"]
)

df["d_occ_similarity"] = (
    1 -
    np.abs(
        df["core_d_occ"] -
        df["shell_d_occ"]
    )
)

df["f_occ_similarity"] = (
    1 -
    np.abs(
        df["core_f_occ"] -
        df["shell_f_occ"]
    )
)

# -------------------------------
# Interface compatibility
# -------------------------------

df["exchange_compatibility"] = (
    df["ofm_cosine"] *
    df["d_occ_similarity"] *
    (1 + df["magmom_product"])
)

df["magnetic_imbalance"] = (
    df["magmom_diff"] *
    df["d_occ_diff"]
)

df["orbital_exchange"] = (
    df["d_occ_product"] *
    df["ofm_cosine"]
)

df["spin_exchange"] = (
    df["magmom_product"] *
    df["d_occ_product"]
)

# -------------------------------
# GS interaction
# -------------------------------

df["gs_exchange"] = (
    df["shell_is_gs"] *
    df["exchange_compatibility"]
)

# -------------------------------
# Density interaction
# -------------------------------

df["density_product"] = (
    df["core_mag_density"] *
    df["shell_mag_density"]
)

df["density_difference"] = (
    np.abs(
        df["core_mag_density"] -
        df["shell_mag_density"]
    )
)

In [9]:
new_cols = [

"magmom_mean",
"magmom_norm_diff",
"magmom_ofm",
"magmom_distance",

"d_occ_product",
"f_occ_product",
"d_occ_similarity",
"f_occ_similarity",

"exchange_compatibility",
"magnetic_imbalance",
"orbital_exchange",
"spin_exchange",

"gs_exchange",

"density_product",
"density_difference"

]

print(df[new_cols].head())

print()

print(df[new_cols].isna().sum())

   magmom_mean  magmom_norm_diff  magmom_ofm  magmom_distance  d_occ_product  \
0     2.491725          0.864872    4.433412         0.614250       0.214286   
1     3.459971          0.063160   11.087673         0.871784       0.214286   
2     1.719265          2.000000    0.000000         0.000000       0.000000   
3     1.719265          2.000000    0.000000         0.000000       0.000000   
4     1.719265          2.000000    0.000000         0.000000       0.000000   

   f_occ_product  d_occ_similarity  f_occ_similarity  exchange_compatibility  \
0            0.0          0.928571               1.0                4.932313   
1            0.0          0.928571               1.0               11.156580   
2            0.0          0.571429               1.0                0.000000   
3            0.0          0.571429               1.0                0.000000   
4            0.0          0.571429               1.0                0.000000   

   magnetic_imbalance  orbital_exchang

In [10]:
df.to_csv(
    "exchange_bias_enhanced_v1.csv",
    index=False
)

print(df.shape)

(979, 83)


In [15]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

# ==========================================
# LOAD DATA
# ==========================================

df = pd.read_csv(
    "exchange_bias_enhanced_v1.csv"
)

TARGET = "coer_oe"

X = df.drop(
    columns=[TARGET]
)

y = df[[TARGET]]

# ==========================================
# SPLIT
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.1,
    random_state=42
)

# ==========================================
# NORMALIZE X TO [0,1]
# ==========================================

x_scaler = MinMaxScaler()

X_train = x_scaler.fit_transform(
    X_train
)

X_test = x_scaler.transform(
    X_test
)

# ==========================================
# NORMALIZE y TO [0,1]
# ==========================================

y_scaler = MinMaxScaler()

y_train = y_scaler.fit_transform(
    y_train
).ravel()

y_test = y_scaler.transform(
    y_test
).ravel()

# ==========================================
# CHECK
# ==========================================

print("X range:",
      X_train.min(),
      X_train.max())

print("y range:",
      y_train.min(),
      y_train.max())

# ==========================================
# MODEL
# ==========================================

rf = RandomForestRegressor(
    n_estimators=500,
    random_state=42,
    n_jobs=-1
)

rf.fit(
    X_train,
    y_train
)

# ==========================================
# PREDICT
# ==========================================

pred = rf.predict(
    X_test
)

# ==========================================
# METRICS IN NORMALIZED SPACE
# ==========================================

r2 = r2_score(
    y_test,
    pred
)

mae = mean_absolute_error(
    y_test,
    pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        pred
    )
)

print("\nRESULTS")
print("R2   :", round(r2,4))
print("MAE  :", round(mae,6))
print("RMSE :", round(rmse,6))

X range: 0.0 1.0000000000000002
y range: 0.0 1.0

RESULTS
R2   : 0.555
MAE  : 0.025398
RMSE : 0.048868


In [14]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import KFold, cross_validate

from sklearn.ensemble import RandomForestRegressor

# =====================================
# LOAD DATA
# =====================================

df = pd.read_csv(
    "exchange_bias_enhanced_v1.csv"
)

TARGET = "coer_oe"

X = df.drop(columns=[TARGET])
y = df[TARGET]

# =====================================
# NORMALIZE TO [0,1]
# =====================================

x_scaler = MinMaxScaler()
X = x_scaler.fit_transform(X)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(
    y.values.reshape(-1,1)
).ravel()

# =====================================
# MODEL
# =====================================

rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

# =====================================
# 5-FOLD CV
# =====================================

cv5 = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores5 = cross_validate(
    rf,
    X,
    y,
    cv=cv5,
    scoring=[
        "r2",
        "neg_mean_absolute_error",
        "neg_root_mean_squared_error"
    ],
    n_jobs=-1
)

print("\n===== 5-FOLD CV =====")

print(
    "R2   :",
    scores5["test_r2"].mean(),
    "±",
    scores5["test_r2"].std()
)

print(
    "MAE  :",
    -scores5["test_neg_mean_absolute_error"].mean()
)

print(
    "RMSE :",
    -scores5["test_neg_root_mean_squared_error"].mean()
)

# =====================================
# 10-FOLD CV
# =====================================

cv10 = KFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

scores10 = cross_validate(
    rf,
    X,
    y,
    cv=cv10,
    scoring=[
        "r2",
        "neg_mean_absolute_error",
        "neg_root_mean_squared_error"
    ],
    n_jobs=-1
)

print("\n===== 10-FOLD CV =====")

print(
    "R2   :",
    scores10["test_r2"].mean(),
    "±",
    scores10["test_r2"].std()
)

print(
    "MAE  :",
    -scores10["test_neg_mean_absolute_error"].mean()
)

print(
    "RMSE :",
    -scores10["test_neg_root_mean_squared_error"].mean()
)


===== 5-FOLD CV =====
R2   : 0.45059822058792526 ± 0.1555428854298769
MAE  : 0.022347396946178866
RMSE : 0.04990325068978378

===== 10-FOLD CV =====
R2   : 0.4712354146680949 ± 0.1568599569218893
MAE  : 0.021878416130857753
RMSE : 0.0481149517349229


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold
from sklearn.model_selection import cross_validate

from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor

from sklearn.metrics import make_scorer
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

# =======================================================
# Load dataset
# =======================================================

df = pd.read_csv("exchange_bias_enhanced_v1.csv")

# =======================================================
# Targets
# =======================================================

targets = {

    "Exchange Bias":"exc_bias_oe",

    "Saturation Magnetization":"sat_em_g",

    "Coercivity":"coer_oe",

    "Curie Temperature":"Tc",

    "Remanence":"mr (emu/g)"
}

drop_cols = [

    "core",

    "shell",

    "formula"
]

# =======================================================
# Metrics
# =======================================================

rmse = make_scorer(

    lambda y, p:
    np.sqrt(mean_squared_error(y,p)),

    greater_is_better=False
)

scoring = {

    "R2":"r2",

    "MAE":"neg_mean_absolute_error",

    "RMSE":rmse
}

# =======================================================
# Models
# =======================================================

models = {

"Random Forest":

RandomForestRegressor(

n_estimators=500,

random_state=42,

n_jobs=-1

),

"XGBoost":

XGBRegressor(

n_estimators=500,

learning_rate=0.03,

max_depth=6,

subsample=0.8,

colsample_bytree=0.8,

objective="reg:squarederror",

random_state=42

)

}

# =======================================================
# Cross Validation
# =======================================================

for target_name,target in targets.items():

    print("="*80)

    print(target_name)

    print("="*80)

    X = df.drop(

        columns=drop_cols+[target],

        errors="ignore"
    )

    y = df[target]

    for folds in [5,10]:

        cv = KFold(

            n_splits=folds,

            shuffle=True,

            random_state=42
        )

        print()

        print(f"{folds}-Fold CV")

        print("-"*40)

        for model_name,model in models.items():

            scores = cross_validate(

                model,

                X,

                y,

                cv=cv,

                scoring=scoring,

                n_jobs=-1

            )

            print()

            print(model_name)

            print(

                f"R2   : "

                f"{scores['test_R2'].mean():.4f}"

                f" ± "

                f"{scores['test_R2'].std():.4f}"

            )

            print(

                f"MAE  : "

                f"{-scores['test_MAE'].mean():.4f}"

                f" ± "

                f"{scores['test_MAE'].std():.4f}"

            )

            print(

                f"RMSE : "

                f"{-scores['test_RMSE'].mean():.4f}"

                f" ± "

                f"{scores['test_RMSE'].std():.4f}"

            )

    print()

Exchange Bias

5-Fold CV
----------------------------------------

Random Forest
R2   : 0.4870 ± 0.0917
MAE  : 184.7787 ± 9.0873
RMSE : 287.7551 ± 18.5699

XGBoost
R2   : 0.5232 ± 0.0919
MAE  : 174.5674 ± 9.9581
RMSE : 277.0113 ± 18.4519

10-Fold CV
----------------------------------------

Random Forest
R2   : 0.5074 ± 0.1549
MAE  : 179.2457 ± 15.8115
RMSE : 278.8137 ± 31.4200

XGBoost
R2   : 0.5449 ± 0.1468
MAE  : 168.3220 ± 18.5485
RMSE : 267.5845 ± 32.1682

Saturation Magnetization

5-Fold CV
----------------------------------------

Random Forest
R2   : -43.1910 ± 86.5177
MAE  : 318.9299 ± 203.2342
RMSE : 2519.9173 ± 1663.3379

XGBoost
R2   : -98.7588 ± 198.2617
MAE  : 278.8529 ± 185.1838
RMSE : 2549.0623 ± 1882.7474

10-Fold CV
----------------------------------------

Random Forest
R2   : -43.0224 ± 127.9687
MAE  : 286.4499 ± 306.7390
RMSE : 1916.9851 ± 2148.0160

XGBoost
R2   : -76.0034 ± 224.2671
MAE  : 297.1371 ± 309.7429
RMSE : 2112.0304 ± 2531.3099

Coercivity

5-Fold CV
--